# Step-by-step implementation
The following are the steps to implement the Multi-Representation Indexing:
  1. Import necessary modules
  2. Set up the OpenAI API key
  3. Load documents and split text
  4. Generate document summaries with LLM
  5. Index with multi-representations
  6. Retrieve documents based on query


## What is Multi-Representation Indexing?

**Multi-Representation Indexing** decouples *what you search over* from *what you return*.

Standard RAG embeds and searches the same raw document chunks you ultimately feed to the
LLM. That's a problem when the raw chunks are long or noisy — a large chunk of text often
doesn't embed as a very sharp semantic vector, so similarity search over full documents
retrieves less precisely.

The fix: index a **compact representation** (an LLM-generated summary) for retrieval, while
keeping the **full original document** around to actually return once that summary is
matched. The two representations are linked by a shared `doc_id`.

- **Vectorstore** (`Chroma`) — embeds and searches only the *summaries* → fast, precise
  semantic matching.
- **Docstore** (`InMemoryByteStore`) — holds the *full original documents*, keyed by
  `doc_id` → nothing is lost for generation.
- **`MultiVectorRetriever`** — ties the two together: it searches the vectorstore, reads off
  the matched summary's `doc_id`, and uses that to fetch the corresponding full document
  from the docstore.

![Multi-Representation Indexing diagram](img/multi_representation_indexing.png)

**Flow in this notebook:**
1. Load & split source documents (`shared_data/*.txt`).
2. Summarize each document chunk with an LLM.
3. Embed the *summaries* into `Chroma`; store the *full documents* in `InMemoryByteStore`,
   both tagged with the same `doc_id`.
4. At query time, `MultiVectorRetriever` searches the summary embeddings, then returns the
   matching full document(s) — not the summary itself.

## 1. Import necessary modules


In [1]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.stores import InMemoryByteStore
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
import uuid
from langchain_core.documents import Document

C:\Users\soura\AppData\Local\Temp\ipykernel_23468\1260073813.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


## 2. Set up the OpenAI API key

In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [ ]:
os.environ["OPENAI_API_KEY"]

In [4]:
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
if OPENAI_API_KEY == "":
    raise ValueError("Please set the OPENAI_API_KEY environment variable")

## 3. Load documents and split text


In [5]:
loaders = [
    TextLoader("../shared_data/blog.langchain.dev_announcing-langsmith_.txt", encoding="utf-8"),
    TextLoader("../shared_data/blog.langchain.dev_automating-web-research_.txt", encoding="utf-8"),
]

docs = []
for loader in loaders:
    docs.extend(loader.load())
text_splitter = RecursiveCharacterTextSplitter(chunk_size=10000)
docs = text_splitter.split_documents(docs)

In [9]:
len(docs)

3

## 4. Generate document summaries with LLM


In [6]:
chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{doc}")
    | ChatOpenAI(model="gpt-3.5-turbo", max_retries=0)
    | StrOutputParser()
)

In [7]:
summaries = chain.batch(docs, {"max_concurrency": 3})

In [8]:
summaries

['LangSmith is a new platform by LangChain that aims to help developers transition from prototyping to production with their LLM-powered applications. The platform offers features for debugging, testing, evaluating, and monitoring LLM applications. LangSmith provides visibility into model inputs and outputs, supports dataset creation and testing, integrates with evaluation modules, and offers monitoring capabilities for tracking system and model performance. The platform has been tested by early design partners and has shown to be beneficial for building complex LLM applications. LangSmith is currently in closed beta, but developers can sign up to try the platform.',
 'The document discusses how LangSmith helps users easily create datasets from existing logs and use them for testing and evaluation, seamlessly connecting logging/debugging workflows to testing/evaluation workflows. Fintual, a Latin American startup, moved their development stack onto LangSmith to build evaluation, testin

## 5. Index with multi-representations


**Set up the two stores and the retriever that links them:**
- `vectorstore` — `Chroma` collection that will hold **summary embeddings** only.
- `store` — `InMemoryByteStore` acting as the **docstore** for full original documents.
- `id_key = "doc_id"` — the metadata field name used to link a summary to its full document.
- `retriever` — `MultiVectorRetriever` wired to both stores via `id_key`.
- `doc_ids` — one UUID per document chunk, to be shared between its summary and its full text.

In [10]:
vectorstore = Chroma(collection_name="summaries", embedding_function=OpenAIEmbeddings())

store = InMemoryByteStore()
id_key = "doc_id"

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)
doc_ids = [str(uuid.uuid4()) for _ in docs]

**Wrap each summary as a `Document`**, tagging it with the matching `doc_id` in its metadata — this is what makes the summary→full-document link retrievable later.

In [11]:
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

**Populate both stores:**
- `retriever.vectorstore.add_documents(summary_docs)` — embeds the summaries into `Chroma`.
- `retriever.docstore.mset(...)` — saves each full original document into the byte store, keyed by its `doc_id`.

In [12]:
retriever.vectorstore.add_documents(summary_docs)
retriever.docstore.mset(list(zip(doc_ids, docs)))

## 6. Retrieve documents based on query

In [13]:
query = "What is LangSmith?"
sub_docs = vectorstore.similarity_search(query)
sub_docs[0]

Document(id='2c181725-7a8f-44c2-a272-c0297dcf7557', metadata={'doc_id': '1e41333c-d8e3-4550-9a00-490ed4590fc1'}, page_content='LangSmith is a new platform by LangChain that aims to help developers transition from prototyping to production with their LLM-powered applications. The platform offers features for debugging, testing, evaluating, and monitoring LLM applications. LangSmith provides visibility into model inputs and outputs, supports dataset creation and testing, integrates with evaluation modules, and offers monitoring capabilities for tracking system and model performance. The platform has been tested by early design partners and has shown to be beneficial for building complex LLM applications. LangSmith is currently in closed beta, but developers can sign up to try the platform.')

In [14]:
retrieved_docs = retriever.invoke(query)

In [15]:
retrieved_docs[0].page_content[0:500]

'URL: https://blog.langchain.dev/announcing-langsmith/\nTitle: Announcing LangSmith, a unified platform for debugging, testing, evaluating, and monitoring your LLM applications\n\nLangChain exists to make it as easy as possible to develop LLM-powered applications.\n\nWe started with an open-source Python package when the main blocker for building LLM-powered applications was getting a simple prototype working. We remember seeing Nat Friedman tweet in late 2022 that there was “not enough tinkering happ'

In [16]:
len(retrieved_docs[0].page_content)

9865